In [3]:
!pip install -U transformers datasets

In [2]:
!pip show transformers

Name: transformers
Version: 4.56.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers


In [4]:
!pip show datasets

Name: datasets
Version: 4.1.1
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: dill, filelock, fsspec, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: torchtune


In [34]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# load train, validation, and test
wnli = load_dataset("glue", "wnli")

wnli

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 635
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 71
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 146
    })
})

In [36]:
# the model should have some background on the data

id2label = {0 : 'not_entailment', 1 : 'entailment'}
label2id = {v : k for k, v in id2label.items()}

print(f'labels: {id2label}')

labels: {0: 'not_entailment', 1: 'entailment'}


In [69]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(id2label), label2id=label2id, id2label=id2label)

parameters = sum(p.shape[0] * p.shape[1] if len(p.shape) > 1 else p.shape[0] for p in model.parameters()) / (10 ** 6)

print(f'parameters: {round(parameters)}M')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


parameters: 109M


In [80]:
def preprocess(batch):
    # you can try different preprocessing setup
    return tokenizer([' '.join(pair) for pair in zip(batch['sentence1'], batch['sentence2'])], truncation=True, padding='max_length', max_length=512)

wnli_tokenized = wnli.map(preprocess, batched=True)
wnli_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])


Map:   0%|          | 0/635 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

In [81]:
import numpy as np

def compute_metrics(b):
    # accuracy
    predictions, gold = b
    # now you have the predictions
    predictions = np.argmax(predictions, axis=1)

    corr = 0
    for i in range(len(gold)):
      if gold[i] == predictions[i]:
         corr += 1

    return {'accuracy': corr / len(gold)}


# you can experiment with different hyperparameters

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    learning_rate=1e-7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    disable_tqdm=False,
    report_to='none',
    load_best_model_at_end=True,
    save_total_limit=1,
    weight_decay=0.01,
    warmup=0.1
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=wnli_tokenized['train'],
    eval_dataset=wnli_tokenized['validation'],
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipython-input-1546008996.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.692200,0.708093,0.507042
2,0.671900,0.708431,0.492958
3,0.664400,0.708481,0.492958


TrainOutput(global_step=240, training_loss=0.6962859312693278, metrics={'train_runtime': 83.3534, 'train_samples_per_second': 22.854, 'train_steps_per_second': 2.879, 'total_flos': 501226560460800.0, 'train_loss': 0.6962859312693278, 'epoch': 3.0})

In [100]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# inference - again with the validation set because the test is not labeled
# you can also calculate the accuracy for the test set
with torch.no_grad():
     test_example = wnli['validation'][0]
     sent1, sent2, y = test_example['sentence1'], test_example['sentence2'], test_example['label']

     print(sent1, '<-->', sent2, '-->', id2label[y])

     inputs = tokenizer(' '.join([sent1, sent2]), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to(device)
     o = torch.argmax(model(**inputs).logits, dim=1)

     print(f'answer: {id2label[o.item()]}')

The drain is clogged with hair. It has to be cleaned. <--> The hair has to be cleaned. --> not_entailment
answer: not_entailment
